# Analyze Study (Binary Classification)

This notebook walks through the post-hoc analysis of a completed Octopus classification study. It covers study validation, performance evaluation, feature selection summaries, and feature importance analysis.

Each section can be run independently once the study is loaded.


In [ ]:
from octopus.analysis import (
    aucroc_plot,
    feature_count_plot,
    feature_frequency_plot,
    get_details,
    load_study_info,
    performance,
    performance_plot,
    selected_features,
    workflow_graph,
)

## Load Study

Provide the path to your study directory. If you pass a name prefix (e.g. `"../studies/my_study"`), `load_study_info` automatically finds the latest timestamped run matching that prefix. The study directory is validated on load -- missing outersplits or workflow task directories raise an error immediately.

In [ ]:
study_info = load_study_info("../studies/arrhythmia_classification")

## Study Details

`get_details` returns a summary of the study configuration (ML type, target metric, number of folds).

`workflow_graph` displays the workflow tasks and their dependencies as a tree. Each node shows the task ID, module type, and description. Branches indicate that multiple tasks depend on the same parent.


In [ ]:
details = get_details(study_info)
details

In [ ]:
workflow = workflow_graph(study_info)
print(workflow)

## Performance

Shows metric scores per outer split for all workflow tasks. Feature selection tasks are skipped automatically. By default the `dev` partition is shown -- use this partition for model selection and hyperparameter decisions to avoid data leakage from the held-out test set. Only use `partition="test"` for final, unbiased evaluation after all decisions have been made. Use `metric` to filter to a single metric, or `task` to select a specific workflow task.


In [ ]:
perf = performance(study_info)
fig = performance_plot(perf)
fig.show()

## Selected Features

`selected_features` reads `selected_features.json` from each outersplit and task directory and returns three tables:

- **feature_table**: number of selected features per outersplit and task (plus a Mean row).
- **frequency_table**: how often each feature was selected across outersplits, sorted descending.
- **raw_table**: the raw feature lists per outersplit and task.


In [ ]:
feature_table, frequency_table = selected_features(study_info)
fig = feature_count_plot(feature_table)
fig.show()

In [ ]:
fig = feature_frequency_plot(frequency_table)
fig.show()

## Test Set Performance

The sections above use the **dev** partition -- these results should guide model selection, hyperparameter tuning, and feature selection decisions. Looking at test scores during these steps introduces data leakage and inflates reported performance.

The **test** partition below provides an unbiased estimate of final model performance. Only look at test results **after all modelling decisions have been made**. If test results lead you to change the model or features, the test set is no longer independent and the reported scores lose their validity.

`OctoTestEvaluator` re-computes metrics on the held-out test data per outersplit.

In [ ]:
from octopus.analysis import OctoTestEvaluator

task_predictor_test = OctoTestEvaluator(study=study_info, task_id=0, result_type="best")

## AUCROC Curves

`aucroc_plot` returns two ROC (Receiver Operating Characteristic) curves computed on the held-out test data:

- **Merged ROC**: Pools predictions from all outersplits into a single curve. Gives one overall view of discriminative performance.
- **Averaged ROC**: Computes a ROC curve per outersplit, then plots the mean with a +/- 1 standard deviation band. Shows how stable the model performs across different data splits.

A curve hugging the top-left corner indicates strong discrimination between classes (AUC close to 1.0). The dashed diagonal represents random guessing (AUC = 0.5). A narrow std band in the averaged plot suggests consistent performance across splits.

In [ ]:
fig_merged, fig_averaged = aucroc_plot(task_predictor_test)
fig_merged.show()
fig_averaged.show()